# Workflow of this pipeline

1) **resave** `nd2` images as `tif`, splitting individual channels

2) create RS-FISH detection settings `Log.txt` file in Fiji (independent of this pipeline)

-> **detect spots** using RS-FISH producing:

    - `/.../detections` folder containing a `csv` with spot information for each provided `tif`
    - `merge.csv` combining all `csvs` in the `detections` folder

-> 2.1) (optionaly) correct chromatic shift using a reference registration file, producing a `*_shift-corrected.csv` file for each input file 

3) **visualise the detections**, creating a `/.../detections/vis` folder containing `png` max projections of images in the spot file (eg. `merge.csv`)

4) **segment nuclei** with cellpose, using `tif`s and a cellpose classifyer (default or costum trained) producing:
    - `/.../segmentation` folder containing 2D / 3D segmentation masks in `.npy` and `.tif.` format
    - `/.../segmentation/vis` folder containing 2D / max projection `.png` visualizations of the segmentation
    
5) **filter spots not in nuclei** and calculate sensitivity based on spots from (3) and segmentation masks from (4)
6) spots in 2 channels are matched based on the closest neighbour and **spot distances calculated**
7) **remove tif folder** to save space

In [ ]:
import json
import os
from glob import glob
import sys
import tifffile
import pandas as pd
import numpy as np
import torch
from pathlib import Path

sys.path.append('/home/stumberger/fish-pipelines/')
from fish_utils.resave import remove_tifs, resave_nd2, resave_auto_nd2
from fish_utils.spot_detection import read_parameters, make_fiji_command, detect_spots, combine_csv, add_sample_info, create_folder, plot_detections
from fish_utils.spot_analysis import add_cell_info, get_sensitivity, detect_spot_pairs
from fish_utils.corrections import correct_chrom_shift

from natsort import natsorted
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from skimage.io import imread

from cellpose import models, io
from cellpose.io import imread
from cellpose import plot

# 1) Split nd2 channels and resave as tif

In [ ]:
# specify the upper level folder, containing the "raw" folder with the images
path = '/data/agl_data/NanoFISH/Gabi/GS094_FbnL_FbnR_2i/'

resave_nd2(path) # nd2 file is 1 stack
# resave_auto_nd2(path) # nd2 file contains several z stakcs (eg fields of view created with auto imaging)

# 2) Spot detection
Before running this, open one of your images in fiji, run RS-FISH on it, and save the `Log.txt` file with the spot detection parameters for each of the spot channels- **IMPORTAINT!**

In [ ]:
path = '/data/agl_data/NanoFISH/Gabi/GS094_FbnL_FbnR_2i/' #upper level experiment folder

detection_settings = ['/data/agl_data/NanoFISH/Gabi/GS093_Fbn-L_Fbn-R/20231005_run1_sd/detections/ch1.txt',
                      '/data/agl_data/NanoFISH/Gabi/GS093_Fbn-L_Fbn-R/20231005_run1_sd/detections/ch2.txt']
channels = [1,2] # which channel to detect spots in (counting starts at 0!)

# run
detect_spots(path,detection_settings,channels,
            macro_path = "/home/stumberger/fish-pipelines/fish_utils/RS_macro_param.ijm",
            fiji_path = "/home/stumberger/tools/Fiji.app/ImageJ-linux64")

# combine all csvs in a folder into 1 and add sample info
combine_csv(path)
add_sample_info(path, info=f"{path}/acquisition_info.json")

# 2.1) Chromatic shift correction (optional)


In [ ]:
# path of folder containing the csv with spot coordinates and where to save corrected files
in_path = Path('/data/agl_data/NanoFISH/Gabi/GS094_FbnL_FbnR_2i/detections/')
out_path = in_path

# which file(s) to transform
csv_string = 'merge.csv' 

# registartion file to correct shift created with the `chromatic_abberation_estimation_elastix` 
# notebook in "aligment" module of image_analysis_recepies
transforms_path = Path('/data/agl_data/NanoFISH/Gabi/sd_chrom_shift_regisitration.json')

# which channel the coordinates should be aligned to
reference_channel = '405-CSU-W1'

### column names of interest
# specify how to find coordinates and the channel in tables
# coordinate_column_names_unit = ['z', 'y', 'x']
# alternatively, if you do not want to use unit columns, set them to None
# coordinate_column_names_unit = None
coordinate_column_names_pixel = ['z', 'y', 'x']
channel_column_name = 'channel'

# pixel size should be zyx-array
pixel_size = [0.3,0.13,0.13]

### channel Rrenaming
# if the channel names in the JSON transform file and the coordinate tables differ
# e.g. if the OC names in NIS were different or images were resaved and just have channel 0, 1, ...,
# we have to rename the channels from the JSON file to match the ones in the table
# the channel alias map should have the form: name in JSON -> name in coordinate tables
channel_aliases = {
    '405 CSU-W1': '405-CSU-W1',
    '488 CSU-W1': '488-CSU-W1',
    '561 CSU-W1': 1,
    '640 CSU-W1': 2
}

correct_chrom_shift(in_path, out_path, csv_string, transforms_path, reference_channel,
                        coordinate_column_names_pixel=coordinate_column_names_pixel, pixel_size = pixel_size,
                        channel_column_name = channel_column_name, channel_aliases = channel_aliases)

# 3) Visualise detections
Create visualisation to check how well the spot detection worked. 

In [ ]:
path = "/data/agl_data/NanoFISH/Gabi/GS094_FbnL_FbnR_2i/" #upper level experiment folder
path_spots= None #path to `merge.csv` file; if None defaults to /.../detections/merge.csv 
out_folder = None #where to save visualisations; if None defaults to /.../detections/vis
channels = [1,2] # which channel to plot spots for (counting starts at 0!)
range_quantiles = (0.02, 0.9999) #if the spots are not well visible you can rescale image intensity

plot_detections(path,channels)

# 4) Segmentation
Segmentation needed for further spot filtering and calculations.

In [ ]:
# to do the segmentaion fast work on the gpu
device = torch.device('cuda:1')
torch.cuda.is_available() # should be "True"

In [ ]:
# model to use
model = models.CellposeModel(model_type = "/scratch/stumberger/segmentation/es_nuclei_3d/models/cellpose_residual_on_style_on_concatenation_off_es_nuclei_3d_2023_10_09_10_49_48.635062", device=device)

# in and output paths
in_path = "/data/agl_data/NanoFISH/Gabi/GS080_Dppa3_Nanog-58_1/"
files = glob(f"{in_path}/tif/*_ch0.tif")
out_path = f"{in_path}/segmentation"

# segmentation parameters
chan = [[0,0]]
diams = 85
min_size = 5000
anisotropy = 2.3 # sampling in xy / sampling in z (eg. 0.13 / 0.3 = 2.3)


#create out directories
os.makedirs(f"{out_path}/vis", exist_ok=True)

# apply to all files
for filename in files:
    
    img = io.imread(filename)
    name = os.path.basename(filename).rsplit(".", 1)[0]
    out = f"{out_path}/{name}.tif"
    
    masks, flows, styles = model.eval(img, 
                                      do_3D=True,
                                      diameter = diams,
                                      min_size = min_size,
                                      anisotropy = anisotropy)

    # save results so you can load in gui
    io.masks_flows_to_seg(img, masks, flows, diams, out)

    # save results as png
    io.save_masks(img, masks, flows, out, tif=True)
    
    # max projection of segmentation for quick visualization
    fig = plt.figure(figsize=(12,5))
    plot.show_segmentation(fig, img.max(axis=0), masks.max(axis=0), flows[0].max(axis=0), channels=chan)
    plt.tight_layout()
    fig.savefig(f"{out_path}/vis/{os.path.basename(out)}.png",dpi=300)
    plt.close(fig)

# 5) Filter spots based on segmentation
Here you can exclude spots outside of nuclei based on the segmentation and calculate the spot sensitivity. Only 2d and 3d segmentation masks supported. You can repeat step 3 after this to visualise only spots inside nuclei. 

In [ ]:
## add cell info to spots ##
path = "/data/agl_data/NanoFISH/Gabi/GS080_Dppa3_Nanog-58_1/" #upper level experiment folder
path_spots = f"{path}/detections/merge.csv" #spots file
masks = glob(f"{path}/segmentation/*.npy") #all segmentation masks (.npy and .png supported)
out = f"{path}/detections/spots_filtered.csv" # where to save spots

#filter=True - exclude spots outside of nuclei
add_cell_info(masks,path_spots,out,filter=False,mask_ending="_seg")

# TODO: if correctig for shift, correct after this, if shift correction wasn't performed on images

In [ ]:
## calculate sensitivity ##
path = "/data/agl_data/NanoFISH/Gabi/GS080_Dppa3_Nanog-58_1/" #upper level experiment folder
path_spots = f"{path}/detections/merge.csv" #spots file unfiltered
masks = glob(f"{path}/segmentation/*.npy") #all segmentation masks (.npy and .png supported)
out = f"{path}/detections/spots_per_cell.csv" # where to save spots

get_sensitivity(masks,path_spots,out,mask_ending="_seg")

# 6) Calculate spot distances
Spots in 2 channels are matched based on the closest neighbour and distances calculated.

In [ ]:
path = "/data/agl_data/NanoFISH/Gabi/GS094_FbnL_FbnR_2i/" #upper level experiment folder
# path_spots = f"{path}/detections/shift_corrected/merge_shift-corrected.csv"
path_spots = f"{path}/detections/merge_shift-corrected.csv"
out_distances = f"{path}/distances.csv"
channels = [1,2] # which channels to match
voxel_size=(0.3, 0.13, 0.13) #sizes of zyx [nm]

detect_spot_pairs(path_spots,out_distances,channels,voxel_size)

# 6.1) Merge all distances.csv together (optional)

In [ ]:
# optional
# concatenate all distances.csv into 1 big file
import os
import pandas as pd
from datetime import datetime

# Define the root folder where you want to start the search
root_folder = '/data/agl_data/NanoFISH/Gabi/'

# Define a list to store the paths of "distances.csv" files
csv_files = []

# Walk through all subdirectories and find "distances.csv" files
for root, dirs, files in os.walk(root_folder):
    for file in files:
        if file == "distances.csv":
            csv_files.append(os.path.join(root, file))

# Check if any "distances.csv" files were found
if not csv_files:
    print("No 'distances.csv' files found.")
else:
    # Create an empty DataFrame to store the concatenated data
    concatenated_data = pd.DataFrame()

    # Iterate through the found files, read and concatenate them
    for csv_file in csv_files:
        df = pd.read_csv(csv_file)
        concatenated_data = pd.concat([concatenated_data, df], ignore_index=True)

    # Get the current date and time to create a unique file name
    current_datetime = datetime.now().strftime("%Y%m%d_%H%M%S")

    # Define the output file path with the current date and time
    output_file = f"{root_folder}/all_distances_{current_datetime}.csv"

    # Save the concatenated data to the output file
    concatenated_data.to_csv(output_file, index=False)

    print(f"Concatenated data saved to {output_file}")


# 7) Remove tif folder 
Please always remove tifs afetr you are done with the analysis to save space.

In [ ]:
folder = "/data/agl_data/../tif/"

remove_tifs(folder)